# List-aware · single-fit fixed-origin follow-up

**Goal:** compare the full-strength CatBoost MAE, CatBoost Huber, CatBoost spatial/advanced, XGBoost spatial, LightGBM spatial, and predefined geometric blends over August–October 2025, **without refitting between months**.

Runs `listaware_fixed_origin_followup.py` (6 model fits total rather than 6 fits per month). The final predeclared blends include the leading 50/35/15 blend combined with CatBoost MAE at 80/20, spatial/advanced CatBoost 70/30, and Huber/MAE 70/30.

All months are previously inspected historical development data, not untouched confirmation. **The last cell can also recompute new blends from your existing expanding-run CSV without training.**

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / "listaware_fixed_origin_followup.py").exists():
    raise FileNotFoundError("Open this root notebook from the repository root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from listaware_fixed_origin_followup import run_followup, ADDITIONAL_BLENDS
print("Root:", ROOT)
display(pd.DataFrame([
    {"blend": name, "weights": ", ".join(f"{member} {w:.0%}" for member,w in specs.items())}
    for name, specs in ADDITIONAL_BLENDS.items()
]))

## 1 · Run a short fixed-origin pilot

This pilot uses the same prior-sales cutoff (August 1, 2025), capped historical training rows and boosting rounds. It **does not** replace the full-fidelity confirmation.

In [ ]:
RUN_PILOT = False           # Set True to run / resume; quick preflight recommended.
PILOT_OUTPUT = Path("outputs/listaware_fixed_origin_pilot")
if RUN_PILOT:
    pilot_pooled, pilot_monthly, pilot_types, pilot_preds = run_followup(
        full=False, output_dir=PILOT_OUTPUT, threads=4, bootstrap_repeats=300
    )
    display(pilot_pooled.round(4))
else:
    print("Pilot disabled; set RUN_PILOT=True to fit six small historical models.")

## 2 · Full-strength fixed-origin test

This evaluates all historical audit homes, fits each model **once** on all prior eligible training sales, and predicts all three audit months without updating the models. Predictions and blended score tables resume from `outputs/listaware_fixed_origin_full/`.

**Choose `RUN_FULL=True` explicitly**; six trained models with full iterations still require nontrivial compute.

In [ ]:
RUN_FULL = False
FULL_OUTPUT = Path("outputs/listaware_fixed_origin_full")
if RUN_FULL:
    pooled, monthly, by_type, preds = run_followup(
        full=True, output_dir=FULL_OUTPUT, threads=4,
        bootstrap_repeats=500
    )
else:
    folder = FULL_OUTPUT / "fixed_weight_followup"
    if (folder / "followup_pooled.csv").is_file():
        pooled = pd.read_csv(folder / "followup_pooled.csv")
        monthly = pd.read_csv(folder / "followup_monthly.csv")
        print("Loaded existing fixed-origin scores.")
    else:
        pooled = monthly = None
        print("Full run disabled. Set RUN_FULL=True when ready.")
if pooled is not None:
    display(pooled[[
        "arm","variant","homes","mdape","mape","p90_ape",
        "delta_mdape_pp","delta_mape_pp","delta_mdape_ci_low",
        "delta_mdape_ci_high",
    ]].round(4))

In [ ]:
import matplotlib.pyplot as plt
if monthly is not None:
    display(monthly.pivot_table(index="variant",columns="month",values="delta_mdape_pp").round(3))
if pooled is not None:
    plot = pooled.sort_values("delta_mdape_pp")
    fig,ax = plt.subplots(figsize=(10, max(4,.38*len(plot))))
    ax.barh(plot.variant,plot.delta_mdape_pp)
    ax.axvline(0,color="gray")
    ax.invert_yaxis()
    ax.set_xlabel("Change in MdAPE vs fixed-origin CatBoost Huber (percentage points)")
    plt.tight_layout()
    plt.show()

## 3 · Zero-training blend analysis of your EXISTING full-strength expanding predictions

If you have already completed `outputs/listaware_full_shortlist/predictions.csv`, this cell calculates the **new predeclared blends immediately**. It doesn't retrain any model and it does not relabel the existing expanding forecasts as fixed-origin predictions.

This is useful for deciding which blend hypotheses merit the expensive fixed-origin check.

In [ ]:
RUN_EXISTING_EXPANDING_ANALYSIS = True
EXPANDING_PREDICTIONS = Path("outputs/listaware_full_shortlist/predictions.csv")
if RUN_EXISTING_EXPANDING_ANALYSIS and EXPANDING_PREDICTIONS.is_file():
    expanding_pooled, expanding_monthly, _, _ = run_followup(
        from_predictions=EXPANDING_PREDICTIONS,
        output_dir=Path("outputs/listaware_existing_expanding_blend_followup"),
        bootstrap_repeats=500,
    )
    display(expanding_pooled[["arm","variant","homes","mdape","mape","delta_mdape_pp"]].round(4))
else:
    print("No expanding CSV found; skip scores-only comparison.")

## Interpretation

Compare the original 50/35/15 blend, CatBoost MAE and the 80/20 blend on exactly the same homes; look for month-to-month consistency. Prefer **fixed-origin** for the harder forward-horizon check. Results from these previously inspected months remain exploratory, even if a paired bootstrap interval excludes zero.